# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
print("Example positive atom:", atom('Is', 3, 2, 4))
print("Example explicit exclusion:", atom('Not', 3, 2, 4))


Example positive atom: Is3_2_4
Example explicit exclusion: Not3_2_4


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
general_kb = build_general_kb(n, box_h, box_w, givens)
definite_kb = build_definite_kb(n, box_h, box_w, givens)
assert isinstance(general_kb, PropKB)
assert isinstance(definite_kb, PropDefiniteKB)
assert all(is_definite_clause(clause) for clause in definite_kb.clauses)
assert len(definite_kb.clauses) == len(set(definite_kb.clauses))
print(f"General KB: {len(general_kb.clauses):,} clauses; "
      f"{len(prop_symbols(associate('&', general_kb.clauses)))} symbols")
print(f"Definite KB: {len(definite_kb.clauses):,} facts/rules")
print("Every definite clause has the required form; no duplicate rules.")


General KB: 10,317 clauses; 729 symbols
Definite KB: 21,171 facts/rules
Every definite clause has the required form; no duplicate rules.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Isolated processes guarantee that interrupting an expensive experiment
# cannot leave the notebook kernel busy. Neither algorithm is replaced.
import subprocess
import sys

experiment_code = r"""
import json, sys, time, resource
from sudoku_solver import build_general_kb, atom
from logic_ import pl_resolution, tt_entails, associate
pool = json.load(open('puzzles.json'))
givens = {tuple(map(int, k.split('_'))): v
          for k, v in pool['puzzles'][0]['givens'].items()}
kb = build_general_kb(pool['n'], pool['box_h'], pool['box_w'], givens)
# Select a given: even a logically immediate query is expensive for these
# unoptimized exhaustive implementations.
(r, c), v = sorted(givens.items())[0]
query = atom('Is', r, c, v)
print(f'Query: {query}; clauses: {len(kb.clauses)}', flush=True)
# Limit additional exposure to a very large pair list. This resource limit
# is available on this POSIX test machine; the wall-clock limit is separate.
try:
    resource.setrlimit(resource.RLIMIT_AS, (768 * 1024**2, 768 * 1024**2))
    print('Address-space guard: 768 MiB', flush=True)
except (ValueError, OSError):
    print('Address-space guard unavailable; using the time limit', flush=True)
t0 = time.perf_counter()
try:
    result = (pl_resolution(kb, query) if sys.argv[1] == 'resolution'
              else tt_entails(associate('&', kb.clauses), query))
    print(f'Completed: {result}; elapsed: {time.perf_counter()-t0:.3f}s', flush=True)
except MemoryError:
    print(f'MemoryError under resource guard after {time.perf_counter()-t0:.3f}s', flush=True)
"""
experiment_results = []
for method in ['resolution', 'truth_table']:
    started = time.perf_counter()
    try:
        completed = subprocess.run([sys.executable, '-c', experiment_code, method],
                                   capture_output=True, text=True, timeout=30)
        output = completed.stdout.strip()
        status = ('memory guard reached' if 'MemoryError' in output else
                  'completed' if completed.returncode == 0 else
                  f'process exited {completed.returncode}')
        if completed.returncode != 0:
            output += '\n' + completed.stderr.strip()
    except subprocess.TimeoutExpired as error:
        output = error.stdout or ''
        if isinstance(output, bytes):
            output = output.decode(errors='replace')
        status = 'stopped at the 30-second wall-clock limit'
    elapsed = time.perf_counter() - started
    experiment_results.append({'method': method, 'status': status, 'seconds': elapsed})
    print(f'\n{method}: {status} ({elapsed:.2f}s including startup)')
    print(output)



resolution: stopped at the 30-second wall-clock limit (30.10s including startup)
Query: Is1_2_3; clauses: 10317
Address-space guard unavailable; using the time limit




truth_table: stopped at the 30-second wall-clock limit (30.00s including startup)
Query: Is1_2_3; clauses: 10317
Address-space guard unavailable; using the time limit



### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


**Observed on this run:** both resolution and truth-table checking reached the 30-second timeout without returning an answer. The optional address-space guard was unavailable on this macOS system, so the subprocess wall-clock timeout was the active limit.

The preceding cell reports actual execution, not estimated timings. Each method runs in its own process with a 30-second wall-clock limit and, where supported, a 768 MiB address-space guard. A resource-limited interruption is **not** a logical `False` result. The query is deliberately a given, so failure to finish cannot be explained by the query being difficult to infer logically.

For this puzzle, the general KB has 10,317 clauses: 81 cell-existence clauses, 2,916 within-cell exclusion clauses, 7,290 distinct peer exclusion clauses, and 30 givens. Adding the negated query for resolution produces 10,318 initial clauses, so the supplied implementation first constructs 53,225,403 clause pairs. Truth-table enumeration instead faces 729 Boolean symbols and up to $2^{729}$ assignments. These are different sources of computational cost. See the exact outcomes printed above and the interpretation in Question 2.


**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# A small, independent example checks the AND/OR behavior before Sudoku.
cyclic_kb = PropDefiniteKB()
for clause in ['B ==> A', 'A ==> B', 'C ==> A', '(A & B) ==> Q', 'C']:
    cyclic_kb.tell(expr(clause))
assert pl_bc_entails(cyclic_kb, expr('Q')) is True
assert pl_bc_entails(cyclic_kb, expr('Missing')) is False
unsupported_cycle = PropDefiniteKB()
for clause in ['A ==> B', 'B ==> A']:
    unsupported_cycle.tell(expr(clause))
assert pl_bc_entails(unsupported_cycle, expr('A')) is False
print('Supported cycle: True; unsupported cycle: False; unknown atom: False.')


Supported cycle: True; unsupported cycle: False; unknown atom: False.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# End-to-end timings include KB construction. Both solvers start fresh.
# Then directly compare the provided FC and our BC on every candidate.
# This exhaustive validation intentionally takes longer than a single solve.
from time import perf_counter
import platform

benchmark_results = []
print('Python', platform.python_version(), '|', platform.system(), platform.machine())
print('Times are one measured run per solver and puzzle, not universal speed claims.')
for index, item in enumerate(puzzle_pool, start=1):
    started = perf_counter()
    solved = solve_full_grid_fc(n, box_h, box_w, item['givens'])
    fc_time = perf_counter() - started
    assert solved == item['solution']

    started = perf_counter()
    solved_bc = solve_full_grid_bc(n, box_h, box_w, item['givens'])
    bc_time = perf_counter() - started
    assert solved_bc == item['solution']

    kb = build_definite_kb(n, box_h, box_w, item['givens'])
    checked = 0
    for r in range(1, n + 1):
        for c in range(1, n + 1):
            for v in range(1, n + 1):
                query = atom('Is', r, c, v)
                expected = item['solution'][r, c] == v
                assert pl_bc_entails(kb, query) == expected, (index, r, c, v, 'BC')
                assert pl_fc_entails(kb, query) == expected, (index, r, c, v, 'FC')
                checked += 1
    benchmark_results.append({'puzzle': index, 'givens': item['given_count'],
                              'fc_seconds': fc_time, 'bc_seconds': bc_time,
                              'candidate_queries': checked})
    print(f"Puzzle {index}: {item['given_count']} givens | "
          f"FC {fc_time:.4f}s | BC {bc_time:.4f}s | "
          f"{checked} candidate checks passed for BOTH algorithms", flush=True)

print('\nSolved first puzzle:')
first_solution = solve_full_grid_fc(n, box_h, box_w, puzzle_pool[0]['givens'])
for r in range(1, n + 1):
    print(' '.join(str(first_solution[r, c]) for c in range(1, n + 1)))
print(f"\nValidated {sum(x['candidate_queries'] for x in benchmark_results):,} "
      'cell/value pairs per algorithm, including every incorrect candidate.')


Python 3.12.10 | Darwin arm64
Times are one measured run per solver and puzzle, not universal speed claims.


Puzzle 1: 30 givens | FC 0.0837s | BC 8.6855s | 729 candidate checks passed for BOTH algorithms


Puzzle 2: 33 givens | FC 0.0825s | BC 7.9578s | 729 candidate checks passed for BOTH algorithms


Puzzle 3: 36 givens | FC 0.0908s | BC 6.5042s | 729 candidate checks passed for BOTH algorithms


Puzzle 4: 39 givens | FC 0.0858s | BC 6.0107s | 729 candidate checks passed for BOTH algorithms


Puzzle 5: 42 givens | FC 0.0878s | BC 4.8144s | 729 candidate checks passed for BOTH algorithms



Solved first puzzle:
1 3 6 9 8 5 4 7 2
2 9 4 3 1 7 8 6 5
5 8 7 2 6 4 1 3 9
7 5 1 4 9 6 2 8 3
3 6 9 8 7 2 5 1 4
4 2 8 5 3 1 7 9 6
8 1 2 6 4 9 3 5 7
6 4 3 7 5 8 9 2 1
9 7 5 1 2 3 6 4 8

Validated 3,645 cell/value pairs per algorithm, including every incorrect candidate.


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Answer:**

Let $I_{rcv}$ mean that cell $(r,c)$ contains $v$, and let $P(r,c)$ be its distinct peers (other cells in the same row, column or box). All indices run from 1 to $n$.

**General representation.** Only `Is` atoms are needed; classical negation is available directly.

1. Cell existence: $\bigvee_{v=1}^{n} I_{rcv}$ for every cell.
2. Cell uniqueness: $\neg I_{rcv}\lor\neg I_{rcw}$ for every cell and $v<w$.
3. Row, column and box uniqueness: $\neg I_{rcv}\lor\neg I_{r'c'v}$ for every value and every unordered pair of peer cells. A peer pair that shares both a row and a box needs only one identical clause.
4. Givens: the unit clause $I_{rcv}$ for each supplied number.

These are already CNF clauses and are added using `PropKB.tell`. A unit contains $n$ cells, each with one of $n$ values; prohibiting repeats therefore makes every value appear exactly once in each unit. The code supports rectangular boxes with `box_h * box_w == n`.

**Definite representation.** A definite clause has **exactly one** positive literal; a Horn clause more generally has **at most one**. The provided `PropDefiniteKB` accepts the definite subset, written as positive premises implying one positive conclusion, or as a positive fact. It cannot accept the cell-existence disjunction above.

I introduce explicit positive atoms $N_{rcv}$ (`Not`) meaning that value $v$ has been excluded. The rule families are:

- Givens: $I_{rcv}$.
- Same-cell exclusion: $I_{rcv}\Rightarrow N_{rcw}$ for $w\ne v$.
- Peer exclusion: $I_{rcv}\Rightarrow N_{r'c'v}$ for $(r',c')\in P(r,c)$.
- Last candidate: $\left(\bigwedge_{w\ne v}N_{rcw}\right)\Rightarrow I_{rcv}$.

For example, learning `Is1_1_3` excludes 3 from its peers. Once eight explicit exclusions are proved in a 9-value cell, its remaining value can be proved. For the degenerate 1-by-1 grid the only value is a fact, since there are no alternatives to exclude.

`Not` is not automatically the classical complement of `Is`. Failing to prove `Is` does not establish `Not`. The last-candidate rule uses the Sudoku requirement that a cell has some value as background justification; it is **not a logically equivalent Horn rewriting** of the general existence clause. This definite KB captures sound elimination deductions on valid puzzles, but can stall on puzzles requiring additional techniques. Completeness of Horn inference is relative to these rules, not to all consequences of the full Sudoku CNF. The solvers report a stalled grid instead of guessing. Both builders reject directly conflicting givens.


### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Answer:**

I would not use either supplied implementation as the default full-grid solver. Soundness and completeness establish the correctness of an unrestricted terminating computation, not a practical time or memory bound.

`tt_entails` recursively assigns a truth value to every proposition. A 9-by-9 grid has $9^3=729$ `Is` symbols, so there are $2^{729}\approx 2.82\times 10^{219}$ assignments in the worst case. The supplied implementation evaluates the KB only at complete assignments and does not use partial-assignment propagation to prune inconsistent branches early. Evaluating a leaf also costs time proportional to the formula size. Its basic depth-first storage is polynomial, but its time is exponential; recursive list/dictionary copies add overhead.

`pl_resolution` instead adds the negated query and constructs all unordered pairs of current clauses. With $m$ clauses this alone takes $\Theta(m^2)$ pair storage and pair visits per round in this implementation. For the first puzzle, its initial 10,318 clauses already require 53,225,403 pairs. Resolvents can add many more clauses and force repeated rounds; the number of possible distinct non-tautological clauses over $k$ atoms is up to $3^k$, so worst-case saturation is exponential. Thus the quadratic work is only the cost of one round, not the entire algorithm's complexity.

The experiment above records a bounded attempt for each method. A timeout or memory-guard termination is evidence about computational resources, not a counterexample to logical completeness. Even querying a given need not finish quickly because these particular implementations do not first perform a direct-fact shortcut. The observed result should be read with the stated 30-second and 768 MiB limits.

Definite rules permit much cheaper propagation. A properly indexed forward engine processes each fact and each premise occurrence a bounded number of times; our wrapper supplies a premise index to the unchanged library routine. This makes the provided elimination-solvable puzzles practical. Modern SAT solvers can also solve Sudoku effectively, but the two unoptimized library procedures are not representative of industrial SAT performance.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Answer:**

My algorithm searches an AND/OR proof graph: a rule succeeds only when **all** its premises succeed; a goal succeeds when **any** rule concluding it succeeds. Facts are the base cases. The following recursive pseudocode expresses the implemented control flow; Python uses explicit `[goal, rule_index, premise_index]` stack frames to avoid the interpreter's recursion-depth limit.

```text
BC(KB, query):
    index rules by their conclusion
    proven = a copy of KB's positive facts
    repeat:
        old_count = size(proven)
        failed_this_round = empty set
        active_path = empty set
        if PROVE(query): return True
        if size(proven) == old_count: return False
        # New proofs may unblock previously cycle-dependent failures.
        # Retry the root with a fresh failed set.

PROVE(goal):
    if goal in proven: return True
    if goal in active_path: return False for this branch
    if goal in failed_this_round: return False for this round
    add goal to active_path
    for premises in rules_whose_conclusion_is(goal):
        if every premise can be PROVE'd (short-circuit AND):
            add goal to proven
            record the supporting premises if tracing
            remove goal from active_path
            return True
    remove goal from active_path
    add goal to failed_this_round
    return False
```

**Soundness.** Only initial facts and heads of rules with already proved premises enter `proven`. By induction every cached success is entailed. Encountering an active goal never counts as success, so a cycle such as `A => B, B => A` cannot create its own evidence.

**Why failures are temporary.** Consider `B => A`, `A => B`, `C => A`, `(A & B) => Q`, and fact `C`. While proving `A`, attempting `B` encounters active `A` and can fail. The alternative through `C` then proves `A`. A permanent negative memo for `B` would wrongly prevent `Q`. Instead, the new proof causes another round, where `B` can now use proved `A`. Positive results remain valid across rounds; failures do not.

**Termination.** Within a round the active-path check prevents repeated goals on a branch, and completed goals are either proved or temporarily failed. Each rule/premise is visited only a bounded number of times per goal expansion. A new round starts only after the finite set `proven` strictly grows. Thus at most the number of KB atoms' worth of productive rounds can occur before success or a final no-progress round. Explicit frames also allow long finite proof chains without `RecursionError`.

**Completeness for finite definite KBs.** Every entailed atom has a finite proof rooted in facts. During a no-progress round, all alternatives for a failed reachable goal have been explored and are blocked by some unproved premise; a cycle with no newly established support cannot supply a finite fact-rooted proof. Conversely, if a proof is blocked only because of a temporary cycle-dependent failure, some shorter supported proof can add a positive atom, causing another round and another opportunity to use it. Therefore stopping on no progress does not discard a remaining fact-rooted proof of the query.

Rule indexes may be reused, but inferred successes and failures are local to each query. The subclass invalidates indexes when `tell` or `retract` changes the KB. There is no call to forward chaining inside backward chaining, and no Sudoku-specific shortcut in the proof engine. Tests cover unsupported cycles, alternative supporting rules, an AND goal needing a retry, unknown atoms, KB mutation, a 1,500-rule chain, and randomly generated cyclic KBs compared with an independent least-model oracle. The tutor mode records the actual successful rules and shows their premises before their conclusions.


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Answer — Naked Pairs:**

Choose a row, column or box $U$, two distinct cells $x,y\in U$, and two distinct values $a,b$. Write $N_{x,v}$ for the explicit exclusion atom for cell $x$ and value $v$. Define a conjunction (notation only, not an extra atom):

$$E_{x,y,a,b}=\bigwedge_{v\notin\{a,b\}}(N_{x,v}\land N_{y,v}).$$

For each other cell $z\in U\setminus\{x,y\}$, add these two definite rules:

$$E_{x,y,a,b}\Rightarrow N_{z,a},\qquad E_{x,y,a,b}\Rightarrow N_{z,b}.$$

Each premise and conclusion is a positive `Not` atom, so these have the required definite-clause form. There is no disjunction and no test for the absence of a proof.

For example, suppose row 1 cells (1,1) and (1,2) have explicit exclusions for all of 1, 2, 4, 5, 6, 8 and 9. Their only remaining possibilities are 3 and 7. In a valid completion they must occupy those two values in some order, because neither can take an outside value and the row prohibits repeats. Therefore every other cell in row 1 excludes both 3 and 7. The rule does not need to guess which cell gets which value. If a candidate were additionally excluded in one pair cell, the deduction is still sound provided the underlying puzzle is consistent.

**Consequences.** These rules can unlock deductions unavailable to basic last-candidate reasoning, while preserving a transparent proof trace. They also increase storage and matching costs substantially. Naively enumerating every unit, cell pair, value pair and target gives

$$3n\binom n2\binom n2\,2(n-2)=O(n^6)$$

rules, each with $2(n-2)$ premises, or $O(n^7)$ premise occurrences. At $n=9$, that is 489,888 raw rules, each with 14 premises, before removing overlaps. Targeted generation or indexing helps, but adding this pattern still does not guarantee a solution to every Sudoku. This illustrates the distinction between what definite clauses can express and the practical size/coverage of a chosen rule set. The submitted basic solver does not add Naked Pairs; this question describes the extension.


### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Answer:**

**(a) When backward chaining can win.** Suppose a large KB has many unrelated rule components, while one query is supported by a few facts and a short chain. With a conclusion index already available, backward chaining visits only that relevant proof. For example, a query about an already-given cell needs no rule expansion after the facts/index are loaded. A forward run can spend time processing unrelated available facts before reaching the desired conclusion. Building an index from scratch still costs a scan of the KB, so this advantage is strongest with reusable indexes and sparse query dependencies; it is not guaranteed by the words “goal-driven” alone.

**(b) When it does not win.** Sudoku's peer dependencies connect much of the board. Proving an incorrect candidate may require exhausting numerous alternatives and revisiting cycle-dependent failures. Solving every cell also asks many closely related queries, so independent backward searches repeat work. Our backward implementation preserves rule indexes but deliberately creates fresh proof tables for each query. A single forward closure, by contrast, shares every deduction across all cells.

The full-grid measurements in Part A include KB construction and final grid validation for both solvers. The forward solver calls the **supplied** `pl_fc_entails` once with a fresh unreachable sentinel atom, forcing its agenda to exhaustion; an indexed `PropDefiniteKB` subclass observes processed facts. It then reconstructs the grid from that closure. The backward solver tries candidate values 1 through n for each cell using `pl_bc_entails`. These are the implemented workloads; the comparison is not a claim that every FC implementation always beats every BC implementation.

The table printed below is generated directly from this run's measurements. The five puzzles are useful workload examples, not a controlled difficulty experiment: clue placement and proof dependencies matter as well as clue count. Timing ratios depend on machine, interpreter, indexes, rule ordering and query reuse.


**Measured outcome:** on this Python 3.12.10 / macOS arm64 run, FC took 0.0825–0.0908 seconds and BC took 4.8144–8.6855 seconds across the five puzzles. For Puzzle 1, the respective times were 0.0837 and 8.6855 seconds (about 103.8 times longer for BC). All five results matched the reference solutions. These measurements support using the shared forward pass for this full-grid workload while retaining backward queries for targeted explanations.


In [8]:
print(f"{'Puzzle':<8}{'Givens':<8}{'FC (s)':>12}{'BC (s)':>12}{'BC / FC':>12}")
for result in benchmark_results:
    print(f"{result['puzzle']:<8}{result['givens']:<8}"
          f"{result['fc_seconds']:>12.4f}{result['bc_seconds']:>12.4f}"
          f"{result['bc_seconds']/result['fc_seconds']:>12.1f}")


Puzzle  Givens        FC (s)      BC (s)     BC / FC
1       30            0.0837      8.6855       103.8
2       33            0.0825      7.9578        96.4
3       36            0.0908      6.5042        71.7
4       39            0.0858      6.0107        70.1
5       42            0.0878      4.8144        54.8


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

**Deployment pending:** the app has been verified locally. A cloud URL will be added after the intended GitHub account is selected, the private repository is created, and Streamlit deployment is verified. The local URL is not a deployed submission URL.


## References and implementation notes

The implementation uses the supplied course utilities unchanged. Reference materials were consulted for concepts and structure; the Sudoku builders, cycle-safe proof control, tests and app were written for this assignment.

- [AIMA Python](https://github.com/aimacode/aima-python): knowledge-base interfaces and the supplied-style inference algorithms.
- [Poole and Mackworth, AIPython](https://artint.info/AIPython/), sections 5.3–5.4: top-down AND/OR proofs and explaining supporting rules. The simple recursive procedure needs additional cycle handling for this assignment.
- [Peter Norvig, Solving Every Sudoku Puzzle](https://www.norvig.com/sudoku.html): units, peers and candidate elimination. Its search algorithm is not used here.
- [tyrelh/sudoku-as-sat](https://github.com/tyrelh/sudoku-as-sat): comparison with a CNF-to-SAT workflow; no external SAT solver is used here.
- [jhrcook/streamlit-sudoku](https://github.com/jhrcook/streamlit-sudoku): example of separating a Streamlit interface from its solver; its Pyomo engine is not used here.

The Streamlit application imports the shared implementation. Its loader retains only givens. `solution` is consulted exclusively by this notebook's validation. A `False` entailment answer means “not proved”; the app separately proves an explicit exclusion before presenting a negative explanation.
